# Exercise 3: scikit-learn in Practice

**Part A** uses the **Wisconsin Breast Cancer** dataset (built into scikit-learn, so no download is needed). It has 569 tumor samples, each described by 30 measurements from a microscope image, and each labeled **malignant** (cancerous) or **benign**. Doctors really do use models like this to *help* (never replace) diagnosis.

**Part B** goes back to the penguins for regression.

Run the setup cell, then answer each question. Solutions: `solutions/ex03_sklearn.ipynb`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target              # 30 feature columns; target: 0 = malignant, 1 = benign
cancer = data.frame.copy()
cancer["diagnosis"] = y.map({0: "malignant", 1: "benign"})
penguins = pd.read_csv("../../data/penguins.csv").dropna()
print(X.shape)
X.head()

## Part A: Classification

**A1.** How many samples are malignant vs benign? *(In this dataset `target` 0 = malignant, 1 = benign.)*

In [ ]:
print(cancer["diagnosis"].value_counts())

**A2.** Split `X` and `y` into train and test sets (25% test, `random_state=0`, `stratify=y`).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

**A3.** What accuracy would you get by **always** guessing the most common class? This is your baseline to beat.

In [ ]:
baseline = (y_test == y_train.mode()[0]).mean()
print(f"Baseline: {baseline:.1%}")

**A4.** Train a `KNeighborsClassifier(n_neighbors=5)` on the raw features, then a pipeline with `StandardScaler()` first. Compare their test accuracies.

In [ ]:
raw = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(X_train, y_train)
print(f"raw:    {raw.score(X_test, y_test):.1%}")
print(f"scaled: {scaled.score(X_test, y_test):.1%}")

**A5.** Use 5-fold cross-validation **on the training set** to choose the best k from 1 to 25 (with scaling). Plot accuracy vs k.

In [ ]:
ks = range(1, 26)
scores = [cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k)),
                          X_train, y_train, cv=5).mean() for k in ks]
plt.plot(ks, scores, marker="o")
plt.xlabel("k")
plt.ylabel("CV accuracy")
plt.show()
best_k = ks[int(np.argmax(scores))]
print("best k:", best_k)

**A6.** Train the best model on all of the training data and show the **confusion matrix** on the test set.
In medicine, which mistake is worse: calling a malignant tumor benign, or a benign tumor malignant? How many of each did your model make?

In [ ]:
best = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=best_k)).fit(X_train, y_train)
pred = best.predict(X_test)
print(f"test accuracy: {best.score(X_test, y_test):.1%}")
names = {0: "malignant", 1: "benign"}
print(pd.crosstab(y_test.map(names), pd.Series(pred, index=y_test.index).map(names),
                  rownames=["true"], colnames=["predicted"]))
# Calling a malignant tumor "benign" (a false negative) is far worse: a cancer goes untreated.
# Real systems often trade some overall accuracy to make that mistake rarer.

**A7.** ⭐ **Overfitting.** Train `DecisionTreeClassifier(max_depth=d, random_state=0)` for d = 1 to 12. Plot **training** accuracy and **test** accuracy on the same chart. What happens as the tree gets deeper?

In [ ]:
depths = range(1, 13)
train_acc, test_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))
plt.plot(depths, train_acc, marker="o", label="train")
plt.plot(depths, test_acc, marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()
# Training accuracy climbs to 100% (the tree memorizes the training data), but test accuracy
# stops improving: that gap is OVERFITTING.

## Part B: Regression (penguins)

**B1.** Predict `body_mass_g` from `flipper_length_mm` and `bill_length_mm` with `LinearRegression`. Use a 20% test split (`random_state=1`). Report the test R² and the average absolute error in grams.

In [ ]:
Xp = penguins[["flipper_length_mm", "bill_length_mm"]]
yp = penguins["body_mass_g"]
Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.2, random_state=1)
reg = LinearRegression().fit(Xp_train, yp_train)
pred = reg.predict(Xp_test)
print(f"R^2: {reg.score(Xp_test, yp_test):.3f}")
print(f"average error: {np.abs(pred - yp_test).mean():.0f} g")

**B2.** ⭐ Which **single** feature (bill length, bill depth, flipper length, or body mass) is best at predicting **species** on its own? Use 5-fold cross-validated accuracy with a scaled k-NN (k=5).

In [ ]:
for feature in ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]:
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
    acc = cross_val_score(model, penguins[[feature]], penguins["species"], cv=5).mean()
    print(f"{feature:<18} {acc:.1%}")